## 0. Run setup once

In [1]:
# Install first if needed:
# pip install nltk spacy scikit-learn numpy torch
# python -m spacy download en_core_web_sm

import re
from collections import Counter
import nltk
import spacy
import numpy as np
import torch
import torch.nn as nn
from nltk import word_tokenize, pos_tag, CFG, PCFG, ChartParser, ViterbiParser
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for data in ["punkt", "punkt_tab", "averaged_perceptron_tagger_eng", "wordnet", "omw-1.4"]:
    nltk.download(data, quiet=True)

nlp = spacy.load("en_core_web_sm")
sentences = [
    "I like natural language processing.",
    "I like machine learning and deep learning.",
    "The boy saw the man with the telescope.",
    "Mr. Rahul bought a laptop in Chennai. He loved it."
]


[nltk_data] Error loading punkt: <urlopen error [WinError 10054] An
[nltk_data]     existing connection was forcibly closed by the remote
[nltk_data]     host>


## UNIT I.1 Text preprocessing, stemming and lemmatization

In [2]:
text = "I'm learning NLP at http://test.com :) Running models is fun."
text = text.lower().replace("i'm", "i am")
text = re.sub(r"http\S+|www\.\S+", "", text)
text = text.replace(":)", " <happy> ").replace(":(", " <sad> ")
tokens = re.findall(r"<happy>|<sad>|[a-z]+", text)

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
print("Tokens:", tokens)
print("Stems:", [stemmer.stem(w) for w in tokens])
print("Lemmas:", [w if w.startswith("<") else lemmatizer.lemmatize(w, pos="v") for w in tokens])


Tokens: ['i', 'am', 'learning', 'nlp', 'at', '<happy>', 'running', 'models', 'is', 'fun']
Stems: ['i', 'am', 'learn', 'nlp', 'at', '<happy>', 'run', 'model', 'is', 'fun']
Lemmas: ['i', 'be', 'learn', 'nlp', 'at', '<happy>', 'run', 'model', 'be', 'fun']


## UNIT I.2 Bigram language model and smoothing

In [3]:
corpus = ["i like nlp", "i like ai", "i enjoy nlp"]
tokenized = [s.split() for s in corpus]
unigrams = Counter(w for sent in tokenized for w in sent)
bigrams = Counter((sent[i], sent[i + 1]) for sent in tokenized for i in range(len(sent) - 1))
context = Counter()
for (w1, w2), count in bigrams.items():
    context[w1] += count

vocab_size = len(unigrams)
a, b = "i", "like"
print("MLE:", bigrams[(a, b)] / context[a])
print("Add-one:", (bigrams[(a, b)] + 1) / (context[a] + vocab_size))
k = 0.5
print("Add-k:", (bigrams[(a, b)] + k) / (context[a] + k * vocab_size))

def sentence_probability(sentence, k=1):
    words = sentence.split()
    p = 1.0
    for w1, w2 in zip(words, words[1:]):
        p *= (bigrams[(w1, w2)] + k) / (context[w1] + k * vocab_size)
    return p

def perplexity(sentence, k=1):
    words = sentence.split()
    probs = [(bigrams[(a, b)] + k) / (context[a] + k * vocab_size) for a, b in zip(words, words[1:])]
    return np.prod(probs) ** (-1 / len(probs))

print("Sentence probability:", sentence_probability("i like nlp"))
print("Perplexity:", perplexity("i like nlp"))


MLE: 0.6666666666666666
Add-one: 0.375
Add-k: 0.45454545454545453
Sentence probability: 0.10714285714285714
Perplexity: 3.0550504633038935


## UNIT I.3 Byte Pair Encoding (BPE), simple example

In [4]:
words_bpe = ["l o w </w>", "l o w e r </w>", "n e w e s t </w>", "w i d e s t </w>"]
for step in range(3):
    pairs = Counter()
    for word in words_bpe:
        symbols = word.split()
        pairs.update(zip(symbols, symbols[1:]))
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    updated = []
    for word in words_bpe:
        symbols, merged, i = word.split(), [], 0
        while i < len(symbols):
            if i + 1 < len(symbols) and (symbols[i], symbols[i + 1]) == best:
                merged.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                merged.append(symbols[i])
                i += 1
        updated.append(" ".join(merged))
    words_bpe = updated
    print("Merge:", best, "->", words_bpe)


Merge: ('l', 'o') -> ['lo w </w>', 'lo w e r </w>', 'n e w e s t </w>', 'w i d e s t </w>']
Merge: ('lo', 'w') -> ['low </w>', 'low e r </w>', 'n e w e s t </w>', 'w i d e s t </w>']
Merge: ('e', 's') -> ['low </w>', 'low e r </w>', 'n e w es t </w>', 'w i d es t </w>']


## UNIT I.4 Tiny neural bigram language model

In [5]:
words = "i like natural language processing".split()
word_to_id = {w: i for i, w in enumerate(words)}
X = torch.tensor([word_to_id[w] for w in words[:-1]])
y = torch.tensor([word_to_id[w] for w in words[1:]])
model = nn.Sequential(nn.Embedding(len(words), 8), nn.Linear(8, len(words)))
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.CrossEntropyLoss()
for _ in range(100):
    optimizer.zero_grad()
    loss = loss_fn(model(X), y)
    loss.backward()
    optimizer.step()
predictions = model(X).argmax(dim=1).tolist()
print("Predicted next words:", [words[i] for i in predictions])


Predicted next words: ['like', 'natural', 'language', 'processing']


## UNIT II.1 POS tagging

In [6]:
print(pos_tag(word_tokenize(sentences[0])))


[('I', 'PRP'), ('like', 'VBP'), ('natural', 'JJ'), ('language', 'NN'), ('processing', 'NN'), ('.', '.')]


## UNIT II.2 CFG parsing: PP attachment ambiguity

In [7]:
cfg = CFG.fromstring("""
S -> NP VP
NP -> Det N | Det N PP
VP -> V NP | VP PP
PP -> P NP
Det -> 'the'
N -> 'boy' | 'man' | 'telescope'
V -> 'saw'
P -> 'with'
""")
parser = ChartParser(cfg)
parse_words = "the boy saw the man with the telescope".split()
for tree in parser.parse(parse_words):
    tree.pretty_print()


                 S                                
      ___________|_______                          
     |                   VP                       
     |            _______|________                 
     |           VP               PP              
     |        ___|___         ____|___             
     NP      |       NP      |        NP          
  ___|___    |    ___|___    |     ___|______      
Det      N   V  Det      N   P   Det         N    
 |       |   |   |       |   |    |          |     
the     boy saw the     man with the     telescope

                 S                            
      ___________|___                          
     |               VP                       
     |        _______|___                      
     |       |           NP                   
     |       |    _______|____                 
     |       |   |   |        PP              
     |       |   |   |    ____|___             
     NP      |   |   |   |        NP          
  ___|

## UNIT II.3 PCFG and Viterbi parsing

In [8]:
pcfg = PCFG.fromstring("""
S -> NP VP [1.0]
NP -> Det N [0.6] | Det N PP [0.4]
VP -> V NP [0.7] | VP PP [0.3]
PP -> P NP [1.0]
Det -> 'the' [1.0]
N -> 'boy' [0.4] | 'man' [0.3] | 'telescope' [0.3]
V -> 'saw' [1.0]
P -> 'with' [1.0]
""")
for tree in ViterbiParser(pcfg).parse(parse_words):
    tree.pretty_print()


                 S                            
      ___________|___                          
     |               VP                       
     |        _______|___                      
     |       |           NP                   
     |       |    _______|____                 
     |       |   |   |        PP              
     |       |   |   |    ____|___             
     NP      |   |   |   |        NP          
  ___|___    |   |   |   |     ___|______      
Det      N   V  Det  N   P   Det         N    
 |       |   |   |   |   |    |          |     
the     boy saw the man with the     telescope



## UNIT II.4 Dependency parsing (spaCy)

In [9]:
doc = nlp(sentences[0])
for token in doc:
    print(token.text, "--", token.dep_, "--> ", token.head.text)


I -- nsubj -->  like
like -- ROOT -->  like
natural -- amod -->  language
language -- compound -->  processing
processing -- dobj -->  like
. -- punct -->  like


## UNIT III.1 WordNet: synsets and similarity

In [10]:
s1 = wordnet.synsets("dog")[0]
s2 = wordnet.synsets("cat")[0]
print("Meaning:", s1.definition())
print("Wu-Palmer similarity:", s1.wup_similarity(s2))


Meaning: a member of the genus Canis (probably descended from the common wolf) that has been domesticated by man since prehistoric times; occurs in many breeds
Wu-Palmer similarity: 0.8571428571428571


## UNIT III.2 Vector space models: Bag of Words and TF-IDF

In [11]:
bow = CountVectorizer().fit_transform(sentences)
tfidf = TfidfVectorizer().fit_transform(sentences)
print("BoW shape:", bow.shape)
print("TF-IDF shape:", tfidf.shape)


BoW shape: (4, 23)
TF-IDF shape: (4, 23)


## UNIT III.3 Similarity measures: cosine similarity and Euclidean distance

In [12]:
from sklearn.metrics.pairwise import euclidean_distances
print("Cosine similarity:", cosine_similarity(tfidf[0], tfidf[1])[0, 0])
print("Euclidean distance:", euclidean_distances(tfidf[0], tfidf[1])[0, 0])


Cosine similarity: 0.11831320437434352
Euclidean distance: 1.3279207774755664


In [19]:
from gensim.models import Word2Vec
w2v = Word2Vec(tokenized, vector_size=20, window=2, min_count=1, sg=1, seed=42)
print("Vector for 'learning' (first 5):", w2v.wv["learning"][:5].round(3))
print("Similar to 'learning':", w2v.wv.most_similar("learning", topn=2))

KeyError: "Key 'and' not present"

## UNIT IV.1 RNN sequence classification (small demonstration)

In [13]:
rnn = nn.RNN(input_size=8, hidden_size=16, batch_first=True)
fc = nn.Linear(16, 2)
x = torch.randn(1, 5, 8)
target = torch.tensor([1])
optimizer = torch.optim.Adam(list(rnn.parameters()) + list(fc.parameters()), lr=0.01)
for _ in range(10):
    optimizer.zero_grad()
    _, hidden = rnn(x)
    logits = fc(hidden[-1])
    loss = nn.CrossEntropyLoss()(logits, target)
    loss.backward()
    optimizer.step()
print("Predicted class:", fc(rnn(x)[1][-1]).argmax(dim=1).item())


Predicted class: 1


In [21]:
rnn = nn.RNN(input_size=8, hidden_size=16, batch_first=True)
fc = nn.Linear(16, 2)

sample_input = torch.randn(1, 5, 8)  # (batch=1, seq_len=5, features=8)
output, hidden = rnn(sample_input)
logits = fc(hidden[-1])
print("RNN Output Logits:", logits.detach().numpy().round(3))

RNN Output Logits: [[ 0.377 -0.293]]


In [24]:
from transformers import pipeline
translator = pipeline("translation_en_to_fr", model="Helsinki-NLP/opus-mt-en-fr")
print("Translation:", translator("I like natural language processing.")[0]["translation_text"])

C:\Users\karth\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyError: "Unknown task translation_en_to_fr, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

## UNIT IV.2 Evaluation metrics: BLEU and classification metrics

In [14]:
reference = [["the", "cat", "sat", "on", "the", "mat"]]
candidate = ["the", "cat", "is", "on", "the", "mat"]
bleu = sentence_bleu(reference, candidate, smoothing_function=SmoothingFunction().method1)
print("BLEU:", round(bleu, 3))
y_true, y_pred = [1, 0, 1, 1], [1, 0, 0, 1]
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall:", recall_score(y_true, y_pred, zero_division=0))
print("F1:", f1_score(y_true, y_pred, zero_division=0))


BLEU: 0.254
Accuracy: 0.75
Precision: 1.0
Recall: 0.6666666666666666
F1: 0.8


In [20]:
doc = nlp("Mr. Rahul bought a laptop. He loved it.")
persons = [e.text for e in doc.ents if e.label_ == "PERSON"]
nouns = [c.text for c in doc.noun_chunks if c.text not in persons and c.root.pos_ != "PRON"]

for token in doc:
    if token.text.lower() in ["he", "she", "him", "her"] and persons:
        print(f"Coreference: '{token.text}' refers to '{persons[-1]}'")
    elif token.text.lower() == "it" and nouns:
        print(f"Coreference: '{token.text}' refers to '{nouns[-1]}'")

Coreference: 'He' refers to 'Rahul'
Coreference: 'it' refers to 'a laptop'


## UNIT V.1 Scaled dot-product attention

In [15]:
Q, K, value_matrix = torch.randn(3, 4), torch.randn(3, 4), torch.randn(3, 4)
weights = torch.softmax((Q @ K.T) / np.sqrt(4), dim=-1)
output = weights @ value_matrix
print("Attention weights:\n", weights)
print("Attention output:\n", output)


Attention weights:
 tensor([[0.1954, 0.1946, 0.6101],
        [0.2293, 0.2980, 0.4728],
        [0.1519, 0.6054, 0.2426]])
Attention output:
 tensor([[-0.3685, -0.1097,  1.6461, -1.2125],
        [-0.1871,  0.0089,  1.4731, -0.9121],
        [-0.3328,  0.5288,  1.0651, -0.2911]])


## UNIT V.2 Named Entity Recognition (NER)

In [16]:
doc = nlp("Rahul bought a laptop in Chennai.")
for entity in doc.ents:
    print(entity.text, entity.label_)


Chennai GPE


In [25]:
sentiment = pipeline("sentiment-analysis")
print("Transformer Sentiment:", sentiment("This lab exam is very easy!"))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 311.62it/s]


Transformer Sentiment: [{'label': 'POSITIVE', 'score': 0.9849307537078857}]


In [27]:
from transformers import pipeline, AutoTokenizer, AutoModel
# 1. BERT Fill-Mask
fill = pipeline("fill-mask", model="bert-base-uncased")
print("BERT Fill-Mask:", fill("Natural language [MASK] is fun.")[0]["sequence"])

# 2. BERT Contextual Embeddings
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")
inputs = tok("I like NLP", return_tensors="pt")
emb = bert(**inputs).last_hidden_state
print("BERT Embedding Shape:", emb.shape)

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 256.01it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT Fill-Mask: natural language learning is fun.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 271.98it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT Embedding Shape: torch.Size([1, 6, 768])
